#  Advanced Python for AI and ML
## Module 1 Notebook: Loading and working with data

**Notebook title:** `loading_data.ipynb`

This notebook introduces practical file and dataset handling in Python. It covers:

1. Common dataset file formats
2. Loading CSV files with `pandas`
3. Loading JSON files with `pandas`
4. Creating and saving DataFrames
5. Generating synthetic data with NumPy
6. Building a regression example using student performance data
7. Converting a regression target into a classification label
8. Generating synthetic datasets with scikit-learn:
   - `make_regression`
   - `make_classification`
   - `make_blobs`

The notebook is intentionally self-contained. It creates small sample files during execution, then loads them back using the same methods students would use with real datasets.

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain why different file formats are used for different data tasks.
2. Load CSV and JSON files into pandas DataFrames.
3. Inspect, manipulate, and save tabular data using pandas.
4. Generate synthetic datasets for experimentation.
5. Train a simple linear regression model using generated student performance data.
6. Convert a numeric target variable into a categorical label for classification.
7. Generate regression, classification, and clustering datasets using scikit-learn.

## 1. Dataset File Formats

A **file format** defines how information is encoded, organized, and stored in a file. Different file formats are useful for different types of data and different computational tasks.

| Format | Common Use | Typical Structure |
|---|---|---|
| CSV | Tabular data | Rows and columns separated by commas |
| JSON | Web APIs, nested records | Key-value pairs and nested objects |
| TXT | Simple text data | Plain text lines |
| XLSX | Spreadsheet data | Worksheets, rows, columns, formulas |
| XML | Hierarchical data exchange | Tags and nested elements |
| HTML | Web pages | Markup structure |
| Images | Computer vision | Pixels and metadata |
| PDF/DOCX | Documents | Formatted text and embedded objects |

In machine learning, the choice of file format can affect readability, preprocessing complexity, storage size, and loading speed.

## 2. Environment Setup

We will use:

- `pandas` for loading, inspecting, manipulating, and saving tabular data
- `numpy` for numeric operations and controlled random data generation
- `matplotlib` for basic visualization
- `scikit-learn` for dataset generation and simple machine learning models

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression, make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Display settings for cleaner notebook output
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 3)

# Create a data folder for the files generated by this notebook.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Data folder is ready: {DATA_DIR.resolve()}")

Data folder is ready: C:\Users\willi\William-Codes\advanced-python\Module-1\data


## 3. Creating a Small Example Dataset

Before loading a CSV or JSON file, we first create a small student dataset and save it to disk.

This is useful pedagogically because students can see the full cycle:

1. Create a DataFrame
2. Save it as CSV and JSON
3. Load it back into Python
4. Inspect and manipulate it

In [3]:
data_source = {
    "student_id": [101, 102, 103, 104, 105, 106],
    "name": ["Ava", "Ben", "Chloe", "Daniel", "Emma", "Farah"],
    "study_hours": [4.5, 2.0, 6.0, 1.5, 5.5, 3.0],
    "attendance_rate": [92, 70, 98, 60, 88, 75],
    "final_grade": [82, 58, 91, 45, 86, 63]
}

students = pd.DataFrame(data_source)
 


In [4]:
# Filter students with final_grade > 50
passed_students = students[students['final_grade'] > 50]
passed_students


,student_id,name,study_hours,attendance_rate,final_grade
0,101,Ava,4.5,92,82
1,102,Ben,2.0,70,58
2,103,Chloe,6.0,98,91
4,105,Emma,5.5,88,86
5,106,Farah,3.0,75,63


In [5]:
# Focused students : Filter students who study more than 5 hours and have attendance rate above 80%

focused_students = students[(students['study_hours'] > 5) & (students['attendance_rate'] > 80)]
focused_students

,student_id,name,study_hours,attendance_rate,final_grade
2,103,Chloe,6.0,98,91
4,105,Emma,5.5,88,86


In [41]:
# Weak students : Filter students who study less than 5 hours and have attendance rate below 75%    
Weak_students = students[(students['study_hours'] < 5) & (students['attendance_rate'] <  75)]
Weak_students

,student_id,name,study_hours,attendance_rate,final_grade
1,102,Ben,2.0,70,58
3,104,Daniel,1.5,60,45


## 4. Saving the Example Dataset as CSV and JSON

A DataFrame can be saved using methods such as:

- `to_csv()`
- `to_json()`

We will save the same data in two formats so that we can practice loading both.

In [ ]:
csv_path = DATA_DIR / "students_sample.csv"
json_path = DATA_DIR / "students_sample.json"

students.to_csv(csv_path, index=False)

students.to_json(
    json_path,
    orient="records",
    indent=4
)

print(f"CSV file saved to:  {csv_path}")
print(f"JSON file saved to: {json_path}")

CSV file saved to:  data\students_sample.csv
JSON file saved to: data\students_sample.json


## 5. Loading CSV Files with `pandas`

The `pd.read_csv()` function loads a CSV file into a DataFrame.

Important arguments include:

| Argument | Purpose |
|---|---|
| `filepath_or_buffer` | Path to the CSV file |
| `sep` | Column separator, default is comma |
| `header` | Row used as column names |
| `names` | Custom column names |
| `index_col` | Column to use as row index |

In most introductory use cases, the file path is enough.

In [8]:
students_from_csv = pd.read_csv(csv_path)

students_from_csv

,student_id,name,study_hours,attendance_rate,final_grade
0,101,Ava,4.5,92,82
1,102,Ben,2.0,70,58
2,103,Chloe,6.0,98,91
3,104,Daniel,1.5,60,45
4,105,Emma,5.5,88,86
5,106,Farah,3.0,75,63


### 5.1 Inspecting the Loaded CSV Data

After loading a dataset, always inspect it before modeling.

Common inspection methods include:

- `.head()`
- `.tail()`
- `.info()`
- `.describe()`
- `.shape`
- `.columns`

In [53]:
print("Shape:", students_from_csv.shape)

print("\nColumns:", list(students_from_csv.columns))

# showing the first 5 rows
display(students_from_csv.head()) 
#show the last 5 rows
display(students_from_csv.tail()) # showing the tail

#show information about the dataframe
print("\nDataFrame info:")
students_from_csv.info()

print("\nSummary statistics:")
display(students_from_csv.describe())
 

Shape: (6, 5)

Columns: ['student_id', 'name', 'study_hours', 'attendance_rate', 'final_grade']


,student_id,name,study_hours,attendance_rate,final_grade
0,101,Ava,4.5,92,82
1,102,Ben,2.0,70,58
2,103,Chloe,6.0,98,91
3,104,Daniel,1.5,60,45
4,105,Emma,5.5,88,86


,student_id,name,study_hours,attendance_rate,final_grade
1,102,Ben,2.0,70,58
2,103,Chloe,6.0,98,91
3,104,Daniel,1.5,60,45
4,105,Emma,5.5,88,86
5,106,Farah,3.0,75,63



DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   student_id       6 non-null      int64  
 1   name             6 non-null      object 
 2   study_hours      6 non-null      float64
 3   attendance_rate  6 non-null      int64  
 4   final_grade      6 non-null      int64  
dtypes: float64(1), int64(3), object(1)
memory usage: 372.0+ bytes

Summary statistics:


,student_id,study_hours,attendance_rate,final_grade
count,6.000,6.000,6.000,6.000
mean,103.500,3.750,80.500,70.833
std,1.871,1.864,14.529,18.192
min,101.000,1.500,60.000,45.000
25%,102.250,2.250,71.250,59.250
50%,103.500,3.750,81.500,72.500
75%,104.750,5.250,91.000,85.000
max,106.000,6.000,98.000,91.000


### 5.2 Example: Loading CSV with an Index Column

If one column uniquely identifies each record, we can load it as the row index.

In [54]:
students_indexed = pd.read_csv(csv_path, index_col="student_id")

students_indexed

,name,study_hours,attendance_rate,final_grade
student_id,,,,
101,Ava,4.5,92,82
102,Ben,2.0,70,58
103,Chloe,6.0,98,91
104,Daniel,1.5,60,45
105,Emma,5.5,88,86
106,Farah,3.0,75,63


## 6. Loading JSON Files with `pandas`

JSON stands for **JavaScript Object Notation**. It is widely used for data exchange between web applications, APIs, and servers.

The `pd.read_json()` function loads JSON data into a DataFrame.

Important arguments include:

| Argument | Purpose |
|---|---|
| `path_or_buf` | Path to the JSON file |
| `typ` | Type of object to recover, usually `"frame"` or `"series"` |
| `orient` | Expected JSON structure, such as `"records"` |

In [11]:
students_from_json = pd.read_json(json_path, orient="records")

students_from_json

,student_id,name,study_hours,attendance_rate,final_grade
0,101,Ava,4.5,92,82
1,102,Ben,2.0,70,58
2,103,Chloe,6.0,98,91
3,104,Daniel,1.5,60,45
4,105,Emma,5.5,88,86
5,106,Farah,3.0,75,63


### 6.1 Checking Whether the CSV and JSON Loads Match

Because both files came from the same DataFrame, the resulting data should be equivalent.

In [55]:
students_from_csv.equals(students_from_json)

True

## 7. Manipulating Imported Data

After loading a dataset, we usually transform it before modeling. Common operations include:

- Selecting columns
- Filtering rows
- Creating new columns
- Sorting
- Grouping or aggregating
- Saving the cleaned result

In [58]:
processed_students = students_from_csv.copy()

# Add a simple pass/fail status based on final grade.
processed_students["status"] = np.where(
    processed_students["final_grade"] >= 50,
    "Pass",
    "Fail"
)



# # Add a derived column.
# processed_students["study_category"] = np.where(
#     processed_students["study_hours"] >= 4,
#     "High Study Time",
#     "Low Study Time"
# )

# # Sort by final grade from highest to lowest.
# processed_students = processed_students.sort_values(
#     by="final_grade",
#     ascending=False
# )

processed_students

,student_id,name,study_hours,attendance_rate,final_grade,status
0,101,Ava,4.5,92,82,Pass
1,102,Ben,2.0,70,58,Pass
2,103,Chloe,6.0,98,91,Pass
3,104,Daniel,1.5,60,45,Fail
4,105,Emma,5.5,88,86,Pass
5,106,Farah,3.0,75,63,Pass


### 7.1 Saving the Processed Data

The transformed DataFrame can now be saved back to disk.

In [59]:
processed_csv_path = DATA_DIR / "students_processed.csv"
processed_json_path = DATA_DIR / "students_processed.json"

processed_students.to_csv(processed_csv_path, index=False)
processed_students.to_json(processed_json_path, orient="records", indent=4)

print(f"Processed CSV saved to:  {processed_csv_path}")
print(f"Processed JSON saved to: {processed_json_path}")

Processed CSV saved to:  data\students_processed.csv
Processed JSON saved to: data\students_processed.json
